In [1]:
# Importando as bibliotecas necessárias
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:
# Importando a base de dados
df = pd.read_csv('data/Superstore.csv',encoding='latin1')
df_vendas = df.copy()

# ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode','Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State','Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category','Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

# Nomes das colunas em português
# ['ID da Linha', 'ID do Pedido', 'Data do Pedido', 'Data de Envio', 'Modo de Envio','ID do Cliente', 'Nome do Cliente', 'Segmento', 'País', 'Cidade', 'Estado','CEP', 'Região', 'ID do Produto', 'Categoria', 'Subcategoria','Nome do Produto', 'Vendas', 'Quantidade', 'Desconto', 'Lucro']


In [3]:
# Removendo colunas que não serão úteis
df_vendas = df_vendas.drop(columns=['Row ID'])

# alterando o tipo da coluna
df_vendas['Ship Date'] = pd.to_datetime(df_vendas['Ship Date'],format='%d-%m-%Y')
df_vendas['Order Date'] = pd.to_datetime(df_vendas['Order Date'], format='%d-%m-%Y')


In [ ]:
display(df_vendas.info())
display(df_vendas.describe())

In [ ]:
# Análise exploratória

# Dimensões iniciais do df_vendas
display(df_vendas.shape)

# Valores nulos
nulos = df_vendas.isnull().sum()
nulos_pct = (nulos / len(df_vendas) * 100).round(2)
df_nulos = pd.DataFrame({'Qtd':nulos, 'Percentual':nulos_pct} ) 

display(df_nulos)

# duplicatas exatas
duplicatas_exatas = df_vendas.duplicated().sum()
display(f'Duplicada exata: {duplicatas_exatas}')

# Removendo a duplicata exata
df_vendas = df_vendas.drop_duplicates()

# Checar duplicatas por Order ID + Product ID (não só linha inteira)
mascara = df_vendas.duplicated(subset=['Order ID', 'Product ID'], keep=False)
duplicatas = mascara.sum()
display(f'Linhas duplicadas por Order ID + Product ID: {duplicatas}')

# Calculando o número total de grupos 'Order ID' + 'Product ID'
suspeitos = df_vendas[mascara]
pares_unicos = suspeitos.groupby(['Order ID', 'Product ID']).ngroups
print(f'{len(suspeitos)} linhas envolvendo {pares_unicos} combinações únicas de Order ID + Product ID')

# Verificação de valores inconsistentes - Vendas ou quantidades negativas não fazem sentido
print(df_vendas[df_vendas['Sales'] <= 0] ) 
print(df_vendas[df_vendas['Quantity'] <= 0] )
print(df_vendas[df_vendas['Discount'] < 0] )


In [7]:
# Criando colunas

"""
- Os valores da coluna Sales já estão com o desconto (Discount) aplicado.
- A coluna total_cost já inclui o desconto (Sales) embutido, então ela reflete o custo por unidade vendida na transação, não necessariamente o custo de aquisição do produto em si, que seria fixo independente do desconto.

"""

# Preço unitário
df_vendas['unit_price'] = df_vendas.apply(
    lambda r: r['Sales'] / r['Quantity'] if r['Discount'] == 1 
    else r['Sales'] / (r['Quantity'] * (1 - r['Discount'])), axis=1
)

df_vendas['unit_price'] = df_vendas['unit_price'].round(2)
# Venada sem desconto
df_vendas['sale_without_discount'] = (df_vendas['unit_price'] * df_vendas['Quantity']).round(2)

# Custo total
df_vendas['total_cost'] = (df_vendas['Sales'] - df_vendas['Profit']).round(2)
# Custo unitário
df_vendas['unit_cost'] = (df_vendas['total_cost'] / df_vendas['Quantity']).round(2)

In [13]:
display(df_vendas.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Order ID               9994 non-null   object        
 1   Order Date             9994 non-null   datetime64[ns]
 2   Ship Date              9994 non-null   datetime64[ns]
 3   Ship Mode              9994 non-null   object        
 4   Customer ID            9994 non-null   object        
 5   Customer Name          9994 non-null   object        
 6   Segment                9994 non-null   object        
 7   Country                9994 non-null   object        
 8   City                   9994 non-null   object        
 9   State                  9994 non-null   object        
 10  Postal Code            9994 non-null   int64         
 11  Region                 9994 non-null   object        
 12  Product ID             9994 non-null   object        
 13  Cat

None

In [ ]:
# Verificando a base após a criação das colunas

display(df_vendas.describe())

# Verificando a existência de valores nulos
display(df_vendas.isnull().sum())

# Verificando a existência de valores infinitos
display(np.isinf(df_vendas.loc[:, 'Sales':]).sum())

print(df_vendas[df_vendas['Sales'] <= 0] ) 
print(df_vendas[df_vendas['Quantity'] <= 0] )
print(df_vendas[df_vendas['Discount'] < 0] )
print(df_vendas[df_vendas['unit_price'] <= 0] ) 
print(df_vendas[df_vendas['sale_without_discount'] <= 0] )
print(df_vendas[df_vendas['total_cost'] <= 0] ) 
print(df_vendas[df_vendas['unit_cost'] <= 0] )

In [9]:
# Exportar a base de dados
caminho = Path.cwd() / 'data'
nome_arquivo = 'Superstore_tratada.csv'
caminho_completo = caminho / nome_arquivo

if Path.exists(caminho_completo):
  df_vendas.to_csv(caminho_completo, sep=',', index=False)
else:
  df_vendas.to_csv(caminho_completo, sep=',', index=False)

## Conclusões da Análise exploratória (Base de dados original)
- A base de dados possui 9994 linhas e 24 colunas.
- A 'Row ID' é uma coluna de índice. Não tem relevancia para a análise (deletada).
- A base de dados não apresenta nenhum valor nulo
- Possui 1 duplicata exata (deletada)
- Possui 15 duplicatas por 'Order ID' + 'Product ID'
  - Após remover a duplicata exata, restaram 14 linhas envolvendo 
    7 combinações únicas de Order ID + Product ID, consideradas 
    itens separados legítimos pois Quantity, Sales e Profit 
    apresentam valores diferentes e proporcionais entre si.
- As colunas 'Sales', 'Quantity' e 'Discount' não apresentam valores 
  inconsistentes — nenhum registro com Sales ou Quantity <= 0, 
  e nenhum Discount negativo.

## Base de dados Tratada
- Após a crição das colunas 'unit_price', 'sale_without_discount',	'total_cost' e	'unit_cost' a base não apresentou nenhum valor inconsistente (menor ou igual a 0)

In [11]:
display(sum(df_vendas['sale_without_discount']))
display(sum(df_vendas['Sales']))

2863935.04

2297200.8603